In [16]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CATEGORIES
from src.utils.io import read_jsonl_sample

CATEGORY = 'young_adult'
cfg = CATEGORIES[CATEGORY]
OUTPUT_DIR = cfg.processed_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Category: {cfg.display_name}')
print(f'Books file: {cfg.books_file}')
print(f'Interactions file: {cfg.interactions_file}')
print(f'Output dir: {OUTPUT_DIR}')

Category: Young Adult
Books file: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\raw\goodreads_books_young_adult.json.gz
Interactions file: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\raw\goodreads_interactions_young_adult.json.gz
Output dir: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\interim\young_adult


---
# Books

In [17]:
SAMPLE_SIZE = 5_000
books_sample = read_jsonl_sample(cfg.books_file, nrows=SAMPLE_SIZE)
print(f'Shape: {books_sample.shape}')
print(f'Columns ({len(books_sample.columns)}):')
print(list(books_sample.columns))

Shape: (5000, 29)
Columns (29):
['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'work_id', 'title', 'title_without_series']


In [18]:
summary_books = pd.DataFrame({
    'dtype': books_sample.dtypes.astype(str),
    'non_null': books_sample.notna().sum(),
    'null_count': books_sample.isna().sum(),
    'null_pct': (books_sample.isna().sum() / len(books_sample) * 100).round(1),
    'n_unique': books_sample.apply(lambda col: col.map(str).nunique()),
    'example': books_sample.iloc[0].astype(str).str[:80],
})
display(summary_books)

,dtype,non_null,null_count,null_pct,n_unique,example
isbn,str,5000,0,0.0,3030,
text_reviews_count,str,5000,0,0.0,452,1
series,object,5000,0,0.0,2300,['147734']
country_code,str,5000,0,0.0,1,US
language_code,str,5000,0,0.0,42,
popular_shelves,object,5000,0,0.0,4313,"[{'count': '1057', 'name': 'to-read'}, {'count..."
asin,str,5000,0,0.0,917,B0056A00P4
is_ebook,str,5000,0,0.0,2,true
average_rating,str,5000,0,0.0,211,4.04
kindle_asin,str,5000,0,0.0,2256,B0056A00P4


In [19]:
books_sample.head(3)

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,,1,[147734],US,,"[{'count': '1057', 'name': 'to-read'}, {'count...",B0056A00P4,true,4.04,B0056A00P4,...,,,,https://www.goodreads.com/book/show/12182387-t...,https://s.gr-assets.com/assets/nophoto/book/11...,12182387,4,285263,"The Passion (Dark Visions, #3)","The Passion (Dark Visions, #3)"
1,,2,[425995],US,,"[{'count': '1010', 'name': 'to-read'}, {'count...",B006KLYIAG,true,3.80,B006KLYIAG,...,,,,https://www.goodreads.com/book/show/20135365-h...,https://s.gr-assets.com/assets/nophoto/book/11...,20135365,5,18450480,Hope's Daughter,Hope's Daughter
2,0698143760,17,[493993],US,,"[{'count': '1799', 'name': 'fantasy'}, {'count...",,true,3.80,,...,3,,2014,https://www.goodreads.com/book/show/21401181-h...,https://images.gr-assets.com/books/1394747643m...,21401181,33,24802827,"Half Bad (Half Life, #1)","Half Bad (Half Life, #1)"


In [26]:
BOOKS_COLUMNS_TO_DROP = [
    'isbn',
    'isbn13',
    'asin',
    'kindle_asin',
    'url',
    'link',
    'image_url',
    'country_code',
    'edition_information',
    'publication_day',
    'publication_month',
    'similar_books',
    'title_without_series',
    'is_ebook',
    'description',
    'publisher',
    'format'
]

In [28]:
books_clean = books_sample.drop(columns=[c for c in BOOKS_COLUMNS_TO_DROP if c in books_sample.columns])

In [29]:
summary_clean = pd.DataFrame({
    'dtype': books_clean.dtypes.astype(str),
    'non_null': books_clean.notna().sum(),
    'null_pct': (books_clean.isna().sum() / len(books_clean) * 100).round(1),
})
display(summary_clean)

,dtype,non_null,null_pct
text_reviews_count,str,5000,0.0
series,object,5000,0.0
language_code,str,5000,0.0
popular_shelves,object,5000,0.0
average_rating,str,5000,0.0
authors,object,5000,0.0
num_pages,str,5000,0.0
publication_year,str,5000,0.0
book_id,str,5000,0.0
ratings_count,str,5000,0.0


In [30]:
books_clean.head(5)

,text_reviews_count,series,language_code,popular_shelves,average_rating,authors,num_pages,publication_year,book_id,ratings_count,work_id,title
0,1,[147734],,"[{'count': '1057', 'name': 'to-read'}, {'count...",4.04,"[{'author_id': '50873', 'role': ''}, {'author_...",,,12182387,4,285263,"The Passion (Dark Visions, #3)"
1,2,[425995],,"[{'count': '1010', 'name': 'to-read'}, {'count...",3.80,"[{'author_id': '5395324', 'role': ''}]",,,20135365,5,18450480,Hope's Daughter
2,17,[493993],,"[{'count': '1799', 'name': 'fantasy'}, {'count...",3.80,"[{'author_id': '7314532', 'role': ''}]",416,2014,21401181,33,24802827,"Half Bad (Half Life, #1)"
3,9,[176160],eng,"[{'count': '7173', 'name': 'to-read'}, {'count...",4.35,"[{'author_id': '293603', 'role': ''}]",,,10099492,152,10800440,Twelfth Grade Kills (The Chronicles of Vladimi...
4,428,[],eng,"[{'count': '9481', 'name': 'to-read'}, {'count...",3.71,"[{'author_id': '4018722', 'role': ''}]",351,2014,22642971,1525,42144295,The Body Electric


In [31]:
from src.utils.io import read_jsonl_chunks
import pyarrow as pa
import pyarrow.parquet as pq

BOOKS_OUTPUT_PATH = OUTPUT_DIR / 'books_clean.parquet'
CHUNKSIZE = 50_000

writer = None
total_rows = 0

for chunk_idx, chunk in enumerate(read_jsonl_chunks(cfg.books_file, chunksize=CHUNKSIZE)):
    chunk = chunk.drop(columns=[c for c in BOOKS_COLUMNS_TO_DROP if c in chunk.columns])
    total_rows += len(chunk)
    
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(BOOKS_OUTPUT_PATH, table.schema, compression='snappy')
    writer.write_table(table)
    print(f'  Chunk {chunk_idx}: {len(chunk):,} rows (total: {total_rows:,})')

if writer:
    writer.close()

  Chunk 0: 50,000 rows (total: 50,000)
  Chunk 1: 43,398 rows (total: 93,398)


In [32]:
metadata = pq.read_metadata(BOOKS_OUTPUT_PATH)
print(f'Rows parquet: {metadata.num_rows:,}')
print(f'Columns: {metadata.num_columns}')
print(f'File size: {BOOKS_OUTPUT_PATH.stat().st_size / 1e6:.1f} MB')

df_check = pd.read_parquet(BOOKS_OUTPUT_PATH, columns=None).head(3)
print(f'\nFinal columns ({len(df_check.columns)}):')
print(list(df_check.columns))
display(df_check)

Rows parquet: 93,398
Columns: 14
File size: 63.4 MB

Final columns (12):
['text_reviews_count', 'series', 'language_code', 'popular_shelves', 'average_rating', 'authors', 'num_pages', 'publication_year', 'book_id', 'ratings_count', 'work_id', 'title']


,text_reviews_count,series,language_code,popular_shelves,average_rating,authors,num_pages,publication_year,book_id,ratings_count,work_id,title
0,1,[147734],,"[{'count': '1057', 'name': 'to-read'}, {'count...",4.04,"[{'author_id': '50873', 'role': ''}, {'author_...",,,12182387,4,285263,"The Passion (Dark Visions, #3)"
1,2,[425995],,"[{'count': '1010', 'name': 'to-read'}, {'count...",3.80,"[{'author_id': '5395324', 'role': ''}]",,,20135365,5,18450480,Hope's Daughter
2,17,[493993],,"[{'count': '1799', 'name': 'fantasy'}, {'count...",3.80,"[{'author_id': '7314532', 'role': ''}]",416,2014,21401181,33,24802827,"Half Bad (Half Life, #1)"


# Interactions

In [33]:
INTER_SAMPLE_SIZE = 10_000
inter_sample = read_jsonl_sample(cfg.interactions_file, nrows=INTER_SAMPLE_SIZE)
print(f'Shape: {inter_sample.shape}')
print(f'Columns ({len(inter_sample.columns)}):')
print(list(inter_sample.columns))

Shape: (10000, 10)
Columns (10):
['user_id', 'book_id', 'review_id', 'is_read', 'rating', 'review_text_incomplete', 'date_added', 'date_updated', 'read_at', 'started_at']


In [34]:
summary_inter = pd.DataFrame({
    'dtype': inter_sample.dtypes.astype(str),
    'non_null': inter_sample.notna().sum(),
    'null_count': inter_sample.isna().sum(),
    'null_pct': (inter_sample.isna().sum() / len(inter_sample) * 100).round(1),
    'n_unique': inter_sample.apply(lambda col: col.map(str).nunique()),
    'example': inter_sample.iloc[0].astype(str).str[:80],
})
display(summary_inter)

,dtype,non_null,null_count,null_pct,n_unique,example
user_id,str,10000,0,0.0,134,8842281e1d1347389f2ab93d60773d4d
book_id,str,10000,0,0.0,5619,18667753
review_id,str,10000,0,0.0,10000,be53fe83a6fc83474052f84692f6e90a
is_read,bool,10000,0,0.0,2,False
rating,int64,10000,0,0.0,6,0
review_text_incomplete,str,10000,0,0.0,813,
date_added,str,10000,0,0.0,9757,Wed Mar 29 00:12:52 -0700 2017
date_updated,str,10000,0,0.0,9766,Wed Mar 29 00:12:52 -0700 2017
read_at,str,10000,0,0.0,1729,
started_at,str,10000,0,0.0,1435,


In [35]:
inter_sample.head(3)

,user_id,book_id,review_id,is_read,rating,review_text_incomplete,date_added,date_updated,read_at,started_at
0,8842281e1d1347389f2ab93d60773d4d,18667753,be53fe83a6fc83474052f84692f6e90a,False,0,,Wed Mar 29 00:12:52 -0700 2017,Wed Mar 29 00:12:52 -0700 2017,,
1,8842281e1d1347389f2ab93d60773d4d,428263,2030f56879ebcc307a4b9cd8c83200e8,False,0,,Mon Mar 27 22:01:42 -0700 2017,Mon Mar 27 22:01:42 -0700 2017,,
2,8842281e1d1347389f2ab93d60773d4d,11387515,2fd3cd1acb30b099c135e358669639da,False,0,,Thu Jan 26 13:35:10 -0800 2017,Thu Jan 26 13:35:10 -0800 2017,,


In [40]:
INTER_COLUMNS_TO_DROP = [
    'review_text_incomplete',
    'review_id',
    'read_at',
    'started_at',
]

In [42]:
inter_clean = inter_sample.drop(columns=[c for c in INTER_COLUMNS_TO_DROP if c in inter_sample.columns])

In [43]:
summary_inter_clean = pd.DataFrame({
    'dtype': inter_clean.dtypes.astype(str),
    'non_null': inter_clean.notna().sum(),
    'null_pct': (inter_clean.isna().sum() / len(inter_clean) * 100).round(1),
})
display(summary_inter_clean)

,dtype,non_null,null_pct
user_id,str,10000,0.0
book_id,str,10000,0.0
is_read,bool,10000,0.0
rating,int64,10000,0.0
date_added,str,10000,0.0
date_updated,str,10000,0.0


In [44]:
inter_clean.head(5)

,user_id,book_id,is_read,rating,date_added,date_updated
0,8842281e1d1347389f2ab93d60773d4d,18667753,False,0,Wed Mar 29 00:12:52 -0700 2017,Wed Mar 29 00:12:52 -0700 2017
1,8842281e1d1347389f2ab93d60773d4d,428263,False,0,Mon Mar 27 22:01:42 -0700 2017,Mon Mar 27 22:01:42 -0700 2017
2,8842281e1d1347389f2ab93d60773d4d,11387515,False,0,Thu Jan 26 13:35:10 -0800 2017,Thu Jan 26 13:35:10 -0800 2017
3,8842281e1d1347389f2ab93d60773d4d,8684868,True,3,Tue Dec 17 13:42:25 -0800 2013,Tue Dec 17 13:47:26 -0800 2013
4,8842281e1d1347389f2ab93d60773d4d,8423493,True,2,Sun Dec 08 01:26:12 -0800 2013,Tue Dec 27 05:37:48 -0800 2016


In [45]:
from src.utils.io import read_jsonl_chunks
import pyarrow as pa
import pyarrow.parquet as pq

INTER_OUTPUT_PATH = OUTPUT_DIR / 'interactions_clean.parquet'
INTER_CHUNKSIZE = 500_000

writer = None
total_rows = 0

for chunk_idx, chunk in enumerate(read_jsonl_chunks(cfg.interactions_file, chunksize=INTER_CHUNKSIZE)):
    chunk = chunk.drop(columns=[c for c in INTER_COLUMNS_TO_DROP if c in chunk.columns])
    total_rows += len(chunk)
    
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(INTER_OUTPUT_PATH, table.schema, compression='snappy')
    writer.write_table(table)
    
    if chunk_idx % 10 == 0:
        print(f'  Chunk {chunk_idx}: {total_rows:,} rows so far')

if writer:
    writer.close()

print(f'   Total rows: {total_rows:,}')

  Chunk 0: 500,000 rows so far
  Chunk 10: 5,500,000 rows so far
  Chunk 20: 10,500,000 rows so far
  Chunk 30: 15,500,000 rows so far
  Chunk 40: 20,500,000 rows so far
  Chunk 50: 25,500,000 rows so far
  Chunk 60: 30,500,000 rows so far
   Total rows: 34,919,254


In [ ]:
metadata = pq.read_metadata(INTER_OUTPUT_PATH)
print(f'Rows parquet: {metadata.num_rows:,}')
print(f'Columns: {metadata.num_columns}')
print(f'File size: {INTER_OUTPUT_PATH.stat().st_size / 1e6:.1f} MB')

df_check = pd.read_parquet(INTER_OUTPUT_PATH).head(3)
print(f'\nFinal columns ({len(df_check.columns)}):')
print(list(df_check.columns))
display(df_check)

Rows parquet: 34,919,254
Columns: 6
File size: 854.2 MB

Final columns (6):
['user_id', 'book_id', 'is_read', 'rating', 'date_added', 'date_updated']


,user_id,book_id,is_read,rating,date_added,date_updated
0,8842281e1d1347389f2ab93d60773d4d,18667753,False,0,Wed Mar 29 00:12:52 -0700 2017,Wed Mar 29 00:12:52 -0700 2017
1,8842281e1d1347389f2ab93d60773d4d,428263,False,0,Mon Mar 27 22:01:42 -0700 2017,Mon Mar 27 22:01:42 -0700 2017
2,8842281e1d1347389f2ab93d60773d4d,11387515,False,0,Thu Jan 26 13:35:10 -0800 2017,Thu Jan 26 13:35:10 -0800 2017
